In [1]:
# Sel 1 — muat data
import pandas as pd
df = pd.read_csv("../data/raw/kredit_sumatera.csv")
df.shape

(32400, 19)

In [2]:
# Sel 2 — empat pertanyaan eksplorasi (Langkah 4)
# P1: seberapa tidak seimbang targetnya?
print("Tingkat gagal bayar:", f"{df.gagal_bayar.mean():.2%}")

# P2: apakah komposisi segmen berubah per tahun?
print(pd.crosstab(df.periode.str[:4], df.segmen, normalize="index").round(3))

# P3: kolom mana yang hilang, dan apakah hilangnya acak?
hilang = df.isna().mean().sort_values(ascending=False)
print(hilang[hilang > 0].round(4))
print(df.groupby("segmen")["rata_saldo_90h"].apply(lambda s: s.isna().mean()).round(3))

# P4: adakah kolom mencurigakan (kebocoran target)?
print(df.groupby("gagal_bayar")["status_restrukturisasi"].apply(lambda s: s.notna().mean()))

Tingkat gagal bayar: 8.30%
segmen   nelayan  pedagang_musiman  pegawai_tetap  petani  wiraswasta
periode                                                              
2022       0.067             0.139          0.421    0.13       0.242
2023       0.067             0.142          0.419    0.13       0.241
2024       0.099             0.171          0.339    0.16       0.231
status_restrukturisasi    0.9479
lama_usaha_tahun          0.0294
rata_saldo_90h            0.0231
dtype: float64
segmen
nelayan             0.108
pedagang_musiman    0.000
pegawai_tetap       0.000
petani              0.105
wiraswasta          0.000
Name: rata_saldo_90h, dtype: float64
gagal_bayar
0    0.000000
1    0.627509
Name: status_restrukturisasi, dtype: float64


In [3]:
# Sel 3 — model garis dasar, pembagian menurut waktu (Langkah 5)
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# fitur turunan: beban angsuran terhadap penghasilan
df["rasio_angsuran"] = df.angsuran_bulanan / df.penghasilan_bulanan

num = ["usia", "tanggungan", "lama_usaha_tahun", "penghasilan_bulanan",
       "ragam_penghasilan", "plafon_diajukan", "tenor_bulan", "bunga_tahunan",
       "rasio_angsuran", "rata_saldo_90h", "jumlah_tunggakan_12b"]
X = df[num].join(pd.get_dummies(df[["segmen", "wilayah", "kanal"]]))
y = df.gagal_bayar

# pembagian menurut waktu, bukan acak
latih = df.periode <= "2023-12"
uji = df.periode > "2024-06"

m = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.06,
                                   max_iter=250, random_state=42)
m.fit(X[latih], y[latih])
skor = m.predict_proba(X[uji])[:, 1]
print("AUC out-of-time:", round(roc_auc_score(y[uji], skor), 4))

AUC out-of-time: 0.7744


In [4]:
# Sel 4 — pembanding: pembagian acak 80:20 (Langkah 6)
from sklearn.model_selection import train_test_split

Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
m2 = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.06,
                                    max_iter=250, random_state=42).fit(Xa, ya)
print("AUC pembagian acak:", round(roc_auc_score(yb, m2.predict_proba(Xb)[:, 1]), 4))

AUC pembagian acak: 0.7631


In [5]:
# Sel 5 — cek kestabilan AUC acak dengan beberapa benih
import numpy as np

auc_acak = []
for s in range(5):
    # pembagian acak dengan benih berbeda
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=s, stratify=y)
    mm = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.06,
                                        max_iter=250, random_state=42).fit(Xa, ya)
    auc_acak.append(roc_auc_score(yb, mm.predict_proba(Xb)[:, 1]))

print("AUC acak per benih:", np.round(auc_acak, 4))
print("Rata-rata:", round(np.mean(auc_acak), 4))

AUC acak per benih: [0.7771 0.7994 0.7785 0.7749 0.7798]
Rata-rata: 0.782
